# LOKLOK Ground-Truth Pipeline — Uber Rider Field Study

**Thesis:** *The Social Media Radio Innovation and Its Impact on University Students' Ratings*
**Researcher:** Sherif Mohamed Abd Elmonem Rashwan (Teaching Assistant, Radio & TV, ACU)
**Framework:** Epistemic Systems Theory (EST) · **Platform:** LOKLOK
**Evidence role:** this notebook turns the **Uber rider field corpus** — 60 real in-car participant observations (SR001–SR060), the raw Likert survey battery, the reconciled Uber statement ledger, and the qualitative ride-diaries — into a single **machine-readable ground-truth package** (quantitative ABM calibration + coded qualitative diaries) that feeds the persona engine and the GMM/Monte-Carlo strands, and is **byte-reproducible** (SHA-256 pinned to the registry).

## What this notebook does, end-to-end

1. **Ingests the primary sources** — `survey_answers_real.csv` (60×24 Likert battery), `realsurveydata(inclusive).csv` (60×8 observation layer), `unified_survey_master_thesis.csv` (60×56 enriched master), the `uber_trips_ledger.csv` (187 ledger rows across 28 driver statements), `uber_ride_reconciliation.csv` (60 observed rides), and `RealStudentsProfiles.txt` (60 ride-diary blocks).
2. **Validates deterministically** — the A1/A2/A3 fidelity battery (row/unique-ID completeness, no missing in the 56-column master, all plausibility ranges) that the thesis cross-check reports as **72 PASS · 0 FAIL · 1 WARN · 7 INFO**.
3. **Builds the refined ABM sample** — removes the 8 exact-duplicate 21-item response vectors (SR029, SR043, SR044, SR046, SR051, SR054, SR057, SR058), keeps the lowest-ID representative, assigns `agent_id` 0–51, and re-verifies Cronbach α (BR .849, SV .856, IQ .963, PB .947).
4. **Emits the quantitative calibration** — `agent_parameters.json` (marginals, item PMFs, categorical distributions, Spearman matrix, group tests, joint empirical PDFs, conditional moments, α), `code_maps.json`, `participants.csv`, `trips_universe.csv` (179 completed trips), `trip_empirics.json` (fare/hour/weekday/product empirics; median EGP 81.93), `rides_matched.csv` (52 one-to-one matched rides), and the exploratory report.
5. **Codes the qualitative layer** — parses the 60 diary profiles into structured records (demographics/backstory, media stance, participant observation, key observations, notable quotes, researcher reflections) and codes every block against the **T1–T6** rider themes and the 9-anchor codebook (BR/SV/IQ/CRIT/PB/ENG/LIST/MOB/SMU) using the governance lexicon, producing an auditable codebook + per-participant qualitative anchors.
6. **Pins the fingerprint** — every emitted file is SHA-256 hashed and checked against the registry values (`survey-52` = `bc136c2d…`, `ride-ledger-179` = `be6adeaa…`, 57-col `refined_master_thesis.csv`, canonical diary profiles) so the output is provably identical to the catalog.

## Hardware this notebook is tuned for

- **CPU-only.** This is a pandas/numpy/scipy pipeline — no GPU, no 4-bit LLM, ~150 MB RAM. Runs on any Colab session.
- Models are **not** loaded; qualitative coding is **lexicon-based and fully deterministic**, matching the governance `qual_codebook` contract.

## Honesty guard (thesis-consistent)

The package is a **primary field-observation ground-truth** of one 60-person cohort (52 uniquely-distinguishable response profiles + 179 reconciled trips), not a probability sample, not a treatment effect, and not a causal claim. The two survey strands (survey responses + in-car observations) are triangulated with the driver-statement ledger; every inference is tagged as observation, coded-lead, or design-implication. Qualitative coding is "quantification of a qualitative primary source" — never "a survey of N beyond the coded text."


In [ ]:
# -------------------------------------------------------------------------
# CELL 1 - Install dependencies (CPU-only, minimal footprint)
# -------------------------------------------------------------------------
!pip install -q --upgrade pip
!pip install -q pandas numpy scipy
print("[OK] Dependencies installed (CPU-only, no GPU required).")


In [ ]:
# -------------------------------------------------------------------------
# CELL 2 - Environment check (CPU is fine; must show pandas/numpy/scipy)
# -------------------------------------------------------------------------
import platform
import numpy as np
import pandas as pd
from scipy import stats

print("python :", platform.python_version())
print("numpy  :", np.__version__)
print("pandas :", pd.__version__)
print("scipy  :", stats.__version__)
print("[OK] Env ready.")


In [ ]:
# -------------------------------------------------------------------------
# CELL 3 - Mount Google Drive (inputs + outputs live here)
# -------------------------------------------------------------------------
from google.colab import drive

drive.mount("/content/drive")
print("[OK] Drive mounted.")


In [ ]:
# -------------------------------------------------------------------------
# CELL 4 - Configuration (edit paths here if needed)
# -------------------------------------------------------------------------
import glob
import json
import os
import re
import hashlib
import time
import warnings
from collections import Counter, OrderedDict

warnings.filterwarnings("ignore")

DRIVE_ROOT = "/content/drive/MyDrive"
RAW_DIR = os.path.join(DRIVE_ROOT, "thesis_uber_ground_truth", "raw")     # upload the 6 primary files here
OUT_DIR = os.path.join(DRIVE_ROOT, "thesis_uber_ground_truth", "output")  # artifacts are written here
REF_DIR = os.path.join(DRIVE_ROOT, "thesis_uber_ground_truth", "reference")  # OPTIONAL holdout for byte-compare
os.makedirs(OUT_DIR, exist_ok=True)

REQUIRED_INPUTS = {
    "survey_answers_real.csv":      "raw Likert battery 60x24",
    "realsurveydata(inclusive).csv": "in-car observation layer 60x8",
    "unified_survey_master_thesis.csv": "enriched 56-col master 60x56",
    "uber_trips_ledger.csv":        "driver-statement trip ledger (187 rows)",
    "uber_ride_reconciliation.csv": "60 observed rides reconciled to statements",
    "RealStudentsProfiles.txt":     "60 participant ride-diary blocks",
}

print("=" * 66)
print("UBER GROUND-TRUTH INPUT MANIFEST")
print("=" * 66)
missing = []
for fname, desc in REQUIRED_INPUTS.items():
    p = os.path.join(RAW_DIR, fname)
    ok = os.path.isfile(p)
    sz = os.path.getsize(p) if ok else 0
    print(f"  {'[OK]' if ok else '[!!]'} {fname:38s} {sz:8d} B - {desc}")
    if not ok:
        missing.append(fname)
if missing:
    raise FileNotFoundError(
        f"Missing primary inputs in {RAW_DIR}: {missing}. Upload the six files "
        f"(the raw ground-truth contents of this thesis's field study) and re-run."
    )
print("[OK] All 6 primary sources present.")


In [ ]:
# -------------------------------------------------------------------------
# CELL 5 - Ingest quantitative primary sources
# -------------------------------------------------------------------------
SRV = os.path.join(RAW_DIR, "survey_answers_real.csv")
INC = os.path.join(RAW_DIR, "realsurveydata(inclusive).csv")
MST = os.path.join(RAW_DIR, "unified_survey_master_thesis.csv")

ans = pd.read_csv(SRV, encoding="utf-8-sig")
inc = pd.read_csv(INC, encoding="utf-8-sig")
mst = pd.read_csv(MST, encoding="utf-8-sig")

print("survey_answers_real      :", ans.shape, "(expect 60 x 24)")
print("realsurveydata(inclusive):", inc.shape, "(expect 60 x 8)")
print("unified_survey_master    :", mst.shape, "(expect 60 x 56)")

# --- cross-file consistency on the shared quantitative core -----------------
quant_shared = [c for c in ans.columns if c not in ("student_id",)]
prod_check = mst[["student_id"] + quant_shared].merge(
    ans.rename(columns={c: "a_" + c for c in quant_shared}),
    on="student_id", how="left")
mism = []
for c in quant_shared:
    bad = int((prod_check[c].astype(str) != prod_check["a_" + c].astype(str)).sum())
    if bad:
        mism.append((c, bad))
print("quant core consistency (survey vs master):", "PASS" if not mism else f"FAIL {mism}")

# --- observation layer: shared non-narrative columns ------------------------
obs_shared = [c for c in inc.columns if c != "student_id"]
obs_check = mst[["student_id"] + obs_shared].merge(
    inc.rename(columns={c: "i_" + c for c in obs_shared}),
    on="student_id", how="left")
mism2 = []
for c in obs_shared:
    bad = int((obs_check[c].astype(str) != obs_check["i_" + c].astype(str)).sum())
    if bad:
        mism2.append((c, bad))
print("observation layer consistency:", "PASS" if not mism2 else f"FAIL {mism2}")

assert mst["student_id"].is_unique, "student_id must be unique in master"
assert len(mst) == 60, "master must hold exactly 60 participants"
print("[OK] Quantitative primary sources ingested.")


In [ ]:
# -------------------------------------------------------------------------
# CELL 6 - Parse the 60 ride-diary profile blocks (qualitative primary source)
# -------------------------------------------------------------------------
DIARY = os.path.join(RAW_DIR, "RealStudentsProfiles.txt")
with open(DIARY, encoding="utf-8-sig", errors="replace") as f:
    diary_text = f.read()

# Blocks are separated by lines of the form  **SRnnn — Full Name**.
# NOTE (operations log): the diary file mixes ID padding (SR001-SR009, SR0010,
# SR011-SR060) while the survey/master files use SR0010/SR0011...
# so every join below uses the NUMERIC TAIL of the ID, never the raw label.
block_re = re.compile(r"\*\*SR(\d+)\s*—\s*([^*]+?)\*\*(.*?)(?=\*\*SR\d+\s*—|\Z)", re.S)
master_tail = {int(re.sub(r"\D", "", str(sid))): sid for sid in mst["student_id"]}
blocks = []
for m in block_re.finditer(diary_text):
    tail = int(m.group(1))
    sid = master_tail.get(tail, "SR%03d" % tail)   # canonical master label via tail
    name = " ".join(m.group(2).split())
    body = m.group(3)
    def grab(label):
        m2 = re.search(r"\*\*%s:\*\*\s*(.*?)(?=\n\s*\*\*[^*]+:\*\*|\Z)" % re.escape(label), body, re.S)
        return " ".join(m2.group(1).split()) if m2 else ""
    blocks.append({
        "student_id": sid,
        "profile_name": name,
        "backstory": grab("Demographics & Backstory"),
        "media_stance": grab("Radio & Media Stance"),
        "profile_observation": grab("Participant Observation"),
    })

profiles = pd.DataFrame(blocks)
print("parsed diary blocks:", len(profiles), "(expect 60)")

# --- linkage: diary name must equal master profile_name ---------------------
name_join = mst[["student_id", "profile_name"]].merge(profiles, on="student_id", how="outer")
bad_name = int((name_join["profile_name_x"].fillna("") != name_join["profile_name_y"].fillna("")).sum())
print("diary <-> master profile_name agreement:", "PASS (0)" if bad_name == 0 else f"MISMATCH {bad_name} rows")
if bad_name:
    print(name_join.loc[name_join["profile_name_x"].fillna("") != name_join["profile_name_y"].fillna(""),
                         ["student_id", "profile_name_x", "profile_name_y"]].to_string(index=False))

assert len(profiles) == 60, "expected exactly 60 diary blocks"
print("[OK] Participant-observation diaries parsed (60/60).")


In [ ]:
# -------------------------------------------------------------------------
# CELL 7 - A1/A2/A3 fidelity validation battery (thesis cross-check)
# -------------------------------------------------------------------------
results = []

def check(name, cond, info="", kind="PASS"):
    results.append({"check": name, "status": "PASS" if cond and kind == "PASS" else ("FAIL" if kind == "PASS" else kind), "info": info})
    print(f"  {('PASS' if cond and kind == 'PASS' else 'FAIL' if kind == 'PASS' else kind):5s}  {name:48s} {info}")

print("=" * 66); print("VALIDATION BATTERY (A1/A2/A3)"); print("=" * 66)
# A1 - frame completeness
check("A1.rows_eq_60", len(mst) == 60, f"rows={len(mst)}")
check("A1.unique_ids_60", mst["student_id"].is_unique and mst["student_id"].nunique() == 60, "IDs unique")
check("A1.ids_in_series", set(mst["student_id"]) == {f"SR{i:03d}" for i in range(1, 61)}, "SR001..SR060")
check("A1.diary_blocks_60", len(profiles) == 60, f"blocks={len(profiles)}")
check("A1.answers_rows_60", len(ans) == 60, f"rows={len(ans)}")
check("A1.obs_rows_60", len(inc) == 60, f"rows={len(inc)}")

# A2 - completeness
na = mst.isna().sum()
check("A2.no_missing_56cols", int((na > 0).sum()) == 0, f"cols_with_na={int((na>0).sum())}")
check("A2.answers_no_missing", int(ans.isna().sum().sum()) == 0, "answers NA=0")
check("A2.obs_no_missing", int(inc.isna().sum().sum()) == 0, "observation NA=0")

# A3 - plausibility ranges (observed distributions)
def in_range(s, lo, hi):
    return int((s >= lo).all()) == 1 and int((s <= hi).all()) == 1
check("A3.age_18_23", in_range(mst["age"], 18, 23), f"obs {mst['age'].min()}-{mst['age'].max()}")
check("A3.ride_dur_5_15", in_range(mst["ride_duration_minutes"], 5, 15), "obs 5-15")
check("A3.traffic_ord_1_3", in_range(mst["traffic_ordinal"], 1, 3), "obs 1-3")
check("A3.interaction_ord_1_4", in_range(mst["interaction_ordinal"], 1, 4), "obs 1-4")
for c in ["BR1", "BR2", "BR3", "BR4"]:
    check(f"A3.range_{c}", in_range(mst[c], 0, 4), f"obs {mst[c].min()}-{mst[c].max()}")
for c in ["SV1", "SV2", "SV3", "SV4", "IQ1", "IQ2", "IQ3"]:
    check(f"A3.range_{c}", in_range(mst[c], 1, 4), f"obs {mst[c].min()}-{mst[c].max()}")
for c in ["PB1", "PB2", "PB3", "PB4", "PB5"]:
    check(f"A3.range_{c}", in_range(mst[c], 1, 7), f"obs {mst[c].min()}-{mst[c].max()}")
check("A3.star_1_5", in_range(mst["star_rating"], 1, 5), f"obs {mst['star_rating'].min()}-{mst['star_rating'].max()}")
check("A3.gender_valid", set(mst["gender"]) <= {"M", "F"}, str(sorted(mst["gender"].unique())))
check("A3.acad_year_valid", set(mst["academic_year"].astype(str)) <= {"1", "2", "3", "4"}, "1..4")
check("A3.faculty_non_empty", int((mst["faculty"].astype(str).str.len() == 0).sum()) == 0, "all filled")

# A3 - qualitative layer presence
check("A3.qual_backstory_filled", mst["backstory"].notna().all() and mst["backstory"].astype(str).str.len().gt(20).all(), "backstory text")
check("A3.qual_stance_filled", mst["media_stance"].astype(str).str.len().gt(15).all(), "media stance text")
check("A3.qual_obs_filled", mst["profile_observation"].astype(str).str.len().gt(30).all(), "observation text")
check("A3.qual_quotes_filled", mst["notable_quotes"].astype(str).str.len().gt(3).all(), "notable quote text")
check("A3.qual_reflections_filled", mst["researcher_reflections"].astype(str).str.len().gt(20).all(), "researcher reflection text")

# A1 - reconciliation & ledger linkage
recon = pd.read_csv(os.path.join(RAW_DIR, "uber_ride_reconciliation.csv"), encoding="utf-8-sig")
ledger = pd.read_csv(os.path.join(RAW_DIR, "uber_trips_ledger.csv"), encoding="utf-8-sig", dtype={"earnings_egp": str, "extra_amounts_egp": str})
check("A1.recon_rows_60", len(recon) == 60, f"rows={len(recon)}")
check("A1.recon_link_60", recon["student_id"].is_unique and set(recon["student_id"]) == set(mst["student_id"]), "60 ride observations")
check("A1.ledger_rows", len(ledger) == 187, f"rows={len(ledger)}")

flat = Counter(r["status"] for r in results)
print("=" * 66)
print(f"RESULT: {flat.get('PASS',0)} PASS · {flat.get('FAIL',0)} FAIL · {flat.get('WARN',0)} WARN")
print("(thesis cross-check: 72 PASS · 0 FAIL · 1 WARN · 7 INFO)")
n_fails = flat.get("FAIL", 0)
print("[OK] Validation battery complete." if n_fails == 0 else f"[!!] {n_fails} checks FAILED")


In [ ]:
# -------------------------------------------------------------------------
# CELL 8 - Refined ABM sample: remove exact-duplicate response vectors
# -------------------------------------------------------------------------
qcols = (["age", "ride_duration_minutes", "traffic_ordinal", "interaction_ordinal"]
         + [f"BR{i}" for i in range(1, 5)]
         + [f"SV{i}" for i in range(1, 5)]
         + [f"IQ{i}" for i in range(1, 4)]
         + [f"PB{i}" for i in range(1, 6)]
         + ["star_rating"])

def num_id(p):
    return int(re.sub(r"\D", "", str(p)))

qv = mst[qcols].astype(str).agg("|".join, axis=1)
cnt = qv.value_counts()
drop = set()
for vec, c in cnt[cnt > 1].items():
    ids = sorted(mst.loc[qv == vec, "student_id"].tolist(), key=num_id)
    for d in ids[1:]:
        drop.add(num_id(d))
removed_ids = sorted(drop)
print("exact-duplicate vectors removed:", removed_ids)
print("expected removed ids:", [29, 43, 44, 46, 51, 54, 57, 58])
assert removed_ids == [29, 43, 44, 46, 51, 54, 57, 58], "removed-ID set drifted from registry"

ref = mst[~mst["student_id"].map(num_id).isin(drop)].copy().reset_index(drop=True)
ref["agent_id"] = np.arange(len(ref))
rd_qv = ref[qcols].astype(str).agg("|".join, axis=1)
n_dup_remaining = int((rd_qv.value_counts() > 1).sum())
print("refined N:", len(ref), "(60 - 8) | dup vectors remaining:", n_dup_remaining)
print("agent_id range: 0..", ref["agent_id"].max())
assert len(ref) == 52 and n_dup_remaining == 0
print("[OK] Refined sample = 52 uniquely-distinguishable participants.")


In [ ]:
# -------------------------------------------------------------------------
# CELL 9 - Scale reliability on the refined sample (Cronbach alpha)
# -------------------------------------------------------------------------
scale_items = {
    "BR": ["BR1", "BR2", "BR3", "BR4_reversed"],
    "SV": ["SV1", "SV2", "SV3", "SV4"],
    "IQ": ["IQ1", "IQ2", "IQ3"],
    "PB": ["PB1", "PB2", "PB3", "PB4", "PB5"],
}
scale_means = {
    "BR": "br_credibility_mean", "SV": "sv_validation_mean",
    "IQ": "iq_information_mean", "PB": "pb_experience_mean"}

def cr(alpha):
    k = alpha.shape[1]
    iv = alpha.var(axis=0, ddof=1)
    tv = alpha.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - iv.sum() / tv)

rel = {}
for name, its in scale_items.items():
    a = cr(ref[its])
    rel[name] = round(a, 3)

print("=" * 66); print("CRONBACH ALPHA (refined n=52)"); print("=" * 66)
expect = {"BR": 0.849, "SV": 0.856, "IQ": 0.963, "PB": 0.947}
for k, v in rel.items():
    mark = "PASS" if abs(v - expect[k]) < 0.001 else "CHECK"
    print(f"  {k}: {v:.3f}  (registry {expect[k]:.3f})  [{mark}]")
assert all(abs(rel[k] - expect[k]) < 0.001 for k in expect), "alpha drifted from registry"
print("[OK] Reliability verified.")


In [ ]:
# -------------------------------------------------------------------------
# CELL 10 - Item/scale descriptives + marginal + categorical distributions
# -------------------------------------------------------------------------
def desc(s):
    return {"n": int(s.count()), "mean": round(float(s.mean()), 3),
            "sd": round(float(s.std(ddof=1)), 3),
            "median": float(s.median()), "min": float(s.min()), "max": float(s.max()),
            "q25": float(s.quantile(.25)), "q75": float(s.quantile(.75)),
            "skew": round(float(stats.skew(s)), 3),
            "kurtosis": round(float(stats.kurtosis(s)), 3)}

item_desc = {c: desc(ref[c]) for c in qcols}
scale_desc = {k: desc(ref[v]) for k, v in scale_means.items()}

NUM_ATTRS = ["age", "ride_duration_minutes", "traffic_ordinal",
             "interaction_ordinal"] + list(scale_means.values()) + ["star_rating"]
CAT_ATTRS = ["gender", "academic_year", "faculty", "faculty_group",
             "university", "university_sector", "age_group",
             "traffic_conditions", "interaction_level", "student_mood",
             "listening_context_label"]

categorical = {}
for c in CAT_ATTRS:
    vc = ref[c].value_counts()
    categorical[c] = {str(k): {"n": int(v), "p": round(float(v / len(ref)), 4)}
                      for k, v in vc.items()}

print("=" * 66); print("SCALE DESCRIPTIVES (refined n=52)"); print("=" * 66)
for k, v in scale_desc.items():
    print(f"  {k:6s} mean={v['mean']:6.3f} sd={v['sd']:6.3f} med={v['median']:5} min={v['min']} max={v['max']}")
print("\nDURATION PMF (min):", {int(k): round(float(v / len(ref)), 4) for k, v in ref["ride_duration_minutes"].value_counts().items()})
print("INTERACTION PMF :", {int(k): round(float(v / len(ref)), 4) for k, v in ref["interaction_ordinal"].value_counts().items()})
print("TRAFFIC PMF     :", {int(k): round(float(v / len(ref)), 4) for k, v in ref["traffic_ordinal"].value_counts().items()})
print("\nKEY CATEGORICALS:")
for c in ["gender", "faculty_group", "university_sector", "student_mood", "interaction_level"]:
    print(f"  {c}: {categorical[c]}")
print("[OK] Descriptives computed.")


In [ ]:
# -------------------------------------------------------------------------
# CELL 11 - Correlations (Spearman) + exploratory group tests
# -------------------------------------------------------------------------
corr = ref[NUM_ATTRS].corr(method="spearman").round(3)

print("=" * 66); print("SPEARMAN CORRELATION MATRIX (n=52)"); print("=" * 66)
print(corr.to_string())

# focused pairwise Spearman battery
spearman = OrderedDict()
for a_, b_ in [("ride_duration_minutes", scale_means["PB"]),
               ("ride_duration_minutes", "interaction_ordinal"),
               ("traffic_ordinal", scale_means["PB"]),
               ("interaction_ordinal", scale_means["PB"]),
               ("interaction_ordinal", "star_rating"),
               ("age", scale_means["PB"])]:
    r = stats.spearmanr(ref[a_], ref[b_])
    spearman[f"{a_}_{b_}"] = round(float(r.statistic), 3)
    spearman[f"{a_}_{b_}_p"] = round(float(r.pvalue), 6)
print("PAIRWISE SPEARMAN:")
for k, v in spearman.items():
    print(f"  {k:44s} {v}")

# Mann-Whitney gender battery
def mw(group_col, target):
    g = ref[group_col].unique()
    if len(g) != 2:
        return None
    a, b = (ref.loc[ref[group_col] == gi, target] for gi in sorted(g, key=str))
    U, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    return {"group_a": str(sorted(g, key=str)[0]), "group_b": str(sorted(g, key=str)[1]),
            "n_a": int(len(a)), "n_b": int(len(b)),
            "mean_a": round(float(a.mean()), 3), "mean_b": round(float(b.mean()), 3),
            "U": float(U), "p": float(p)}

def kw(group_col, target):
    groups = [ref.loc[ref[group_col] == gi, target].values
              for gi in ref[group_col].dropna().unique()]
    if len(groups) < 2:
        return None
    H, p = stats.kruskal(*groups)
    means = {str(gi): round(float(ref.loc[ref[group_col] == gi, target].mean()), 3)
             for gi in ref[group_col].unique()}
    return {"groups": means, "H": float(H), "p": float(p), "k": len(groups)}

tests = OrderedDict()
for target in [scale_means["BR"], scale_means["SV"], scale_means["IQ"],
               scale_means["PB"], "star_rating", "ride_duration_minutes"]:
    tests[f"gender_x_{target}"] = mw("gender", target)
for gc in ["academic_year", "faculty_group", "university_sector", "faculty"]:
    for target in [scale_means["PB"], "star_rating"]:
        r = kw(gc, target)
        if r:
            tests[f"{gc}_x_{target}"] = r
print("\nGROUP TESTS (p-values):")
for k, v in tests.items():
    if v:
        print(f"  {k:38s} p={v['p']:.4f}")
print("[OK] Correlation + hypothesis battery done.")


In [ ]:
# -------------------------------------------------------------------------
# CELL 12 - Joint empirical PDFs + conditional moments (agent calibration)
# -------------------------------------------------------------------------
def crosstab(c1, c2):
    tab = pd.crosstab(ref[c1], ref[c2], normalize="index")
    return {str(k): {str(cc): round(float(v), 4) for cc, v in row.items()}
            for k, row in tab.iterrows()}

joints = OrderedDict()
for a_, b_ in [("gender", "faculty_group"),
               ("gender", "university_sector"),
               ("faculty_group", "university_sector"),
               ("academic_year", "faculty_group"),
               ("interaction_ordinal", "ride_duration_minutes"),
               ("traffic_ordinal", "ride_duration_minutes"),
               ("gender", "ride_duration_minutes")]:
    joints[f"{a_}_x_{b_}"] = crosstab(a_, b_)

cond = OrderedDict()
for gc in ["gender", "academic_year", "faculty_group", "university_sector",
           "interaction_ordinal", "traffic_ordinal"]:
    cond[gc] = {}
    for tgt in [scale_means["BR"], scale_means["SV"], scale_means["IQ"],
                scale_means["PB"], "star_rating", "ride_duration_minutes"]:
        grp = {}
        for gi, sub in ref.groupby(gc):
            grp[str(gi)] = {"mean": round(float(sub[tgt].mean()), 3),
                            "sd": round(float(sub[tgt].std(ddof=1)), 3),
                            "n": int(len(sub))}
        cond[gc][tgt] = grp

print("JOINT TABLES:", list(joints.keys()))
print("CONDITIONAL MOMENT GROUPS:", list(cond.keys()))
print("example joint gender_x_faculty_group:")
print(json.dumps(joints["gender_x_faculty_group"], ensure_ascii=False, indent=1))
print("[OK] Joint/conditional calibration computed.")


In [ ]:
# -------------------------------------------------------------------------
# CELL 13 - agent_parameters.json + code_maps.json + participants.csv
# -------------------------------------------------------------------------
def j(*o):
    obj = OrderedDict()
    for k, v in o:
        if isinstance(v, dict):
            obj[k] = OrderedDict(sorted(v.items()))
        elif isinstance(v, list):
            obj[k] = v
        else:
            obj[k] = v
    return obj

def to_json(name, obj):
    with open(os.path.join(OUT_DIR, name), "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

# --- marginal PMFs
duration_pmf = {int(k): round(float(v / len(ref)), 4)
                for k, v in ref["ride_duration_minutes"].value_counts().items()}
inter_pmf = {int(k): round(float(v / len(ref)), 4)
             for k, v in ref["interaction_ordinal"].value_counts().items()}
traffic_pmf = {int(k): round(float(v / len(ref)), 4)
               for k, v in ref["traffic_ordinal"].value_counts().items()}
item_pmf = {c: {str(int(k)): round(float(v / len(ref)), 4)
                for k, v in sorted(ref[c].value_counts().items())}
            for c in qcols}

agent_params = {
    "n_agents": int(len(ref)),
    "removed_for_duplication": removed_ids,
    "duplicate_vectors_after_refinement": int(n_dup_remaining),
    "marginals_numeric": {c: desc(ref[c]) for c in NUM_ATTRS},
    "empirical_pmf": {
        "ride_duration_minutes": dict(duration_pmf),
        "interaction_ordinal": dict(inter_pmf),
        "traffic_ordinal": dict(traffic_pmf),
        "item_responses": item_pmf,
    },
    "categorical": categorical,
    "correlations_spearman": corr.astype(str).to_dict(),
    "conditional_moments": cond,
    "joint_distributions": joints,
    "reliability_cronbach_alpha": rel,
}
to_json("agent_parameters.json", agent_params)

# --- code maps
code_maps = {c: {str(k): int(i) for i, k in enumerate(sorted(ref[c].unique()))}
             for c in ["gender", "academic_year", "faculty", "faculty_group",
                       "university", "university_sector",
                       "traffic_ordinal", "interaction_ordinal"]}
to_json("code_maps.json", code_maps)

# --- lean participants table (39 cols, engine-ready)
lean = pd.DataFrame({
    "agent_id": ref["agent_id"],
    "student_id": ref["student_id"],
    **{f"code_{c}": ref[c].astype(str).map(code_maps[c]) for c in
       ["gender", "academic_year", "faculty", "faculty_group", "university",
        "university_sector"]},
    "gender_label": ref["gender_label"],
    "faculty_group_label": ref["faculty_group"],
    "sector_label": ref["university_sector"],
    "mood": ref["student_mood"],
    "interaction_label": ref["interaction_level"],
    "age": ref["age"],
    "age_group": ref["age_group"],
    "ride_duration_minutes": ref["ride_duration_minutes"],
    "traffic_ordinal": ref["traffic_ordinal"],
    "interaction_ordinal": ref["interaction_ordinal"],
    "br_score": ref["br_credibility_mean"],
    "sv_score": ref["sv_validation_mean"],
    "iq_score": ref["iq_information_mean"],
    "pb_score": ref["pb_experience_mean"],
    "star_rating": ref["star_rating"],
    "BR1": ref["BR1"], "BR2": ref["BR2"], "BR3": ref["BR3"],
    "BR4_reversed": ref["BR4_reversed"],
    "SV1": ref["SV1"], "SV2": ref["SV2"], "SV3": ref["SV3"], "SV4": ref["SV4"],
    "IQ1": ref["IQ1"], "IQ2": ref["IQ2"], "IQ3": ref["IQ3"],
    "PB1": ref["PB1"], "PB2": ref["PB2"], "PB3": ref["PB3"],
    "PB4": ref["PB4"], "PB5": ref["PB5"],
})
lean.to_csv(os.path.join(OUT_DIR, "participants.csv"), index=False, encoding="utf-8-sig")
print("agent_parameters.json: n_agents =", agent_params["n_agents"])
print("participants.csv     :", lean.shape, "(expect 52 x 39)")
print("[OK] Agent calibration artifacts emitted.")


In [ ]:
# -------------------------------------------------------------------------
# CELL 14 - Trip ledger -> trips_universe.csv + trip_empirics.json
# -------------------------------------------------------------------------
LEDGER = os.path.join(RAW_DIR, "uber_trips_ledger.csv")
RECON = os.path.join(RAW_DIR, "uber_ride_reconciliation.csv")
L = pd.read_csv(LEDGER, dtype={"earnings_egp": str, "extra_amounts_egp": str})

TRIP_PRODUCTS = {"UberX Saver", "UberX", "UberX Priority", "Uber Priority"}
L["is_trip"] = L["event_type"].isin(TRIP_PRODUCTS)
trips = L[L["is_trip"]].copy()
print("non-trip ledger rows:", int((~L["is_trip"]).sum()), "(bank/validation/promo/failed)")
print("trip rows:", len(trips), "(expect 179)")

def mk_gap(r):
    try:
        g = float(r["processed_minutes"]) - float(r["trip_minutes"])
    except (TypeError, ValueError):
        return None
    return g if g >= 0 else g + 1440

trips["gap_min"] = trips.apply(mk_gap, axis=1)
trips["earn_egp"] = trips["earnings_egp"].map(
    lambda x: 0.0 if pd.isna(x) else float(str(x).split(";")[0].replace(",", "")))
trips["hour"] = trips["trip_minutes"].map(lambda x: int(float(x) // 60))
trips["weekday"] = ""

wd_from = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"}
MON = {mm: i + 1 for i, mm in enumerate(
    ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])}

def _trip_year(statement_period):
    mm = re.search(r"(\d{4})", str(statement_period))
    return int(mm.group(1)) if mm else None

import datetime as _dt
for idx, r in trips.iterrows():
    try:
        yr = _trip_year(r["statement_period"])
        d = _dt.date(yr, MON[str(r["trip_month"])], int(r["trip_day"]))
        trips.at[idx, "weekday"] = wd_from[d.weekday()]
    except Exception:
        trips.at[idx, "weekday"] = ""

fare_band = pd.cut(trips["earn_egp"], [0, 40, 70, 100, 150, 1000])
fare_dist = {f"{int(b.left)}-{int(b.right)}": {"n": int(v), "p": round(v / len(trips), 4)}
             for b, v in trips["earn_egp"].groupby(fare_band).count().items()}
hour_dist = {int(h): int(v) for h, v in trips["hour"].value_counts().items()}
wd_dist = {str(k): int(v) for k, v in trips["weekday"].value_counts().items()}
prod_dist = {str(k): int(v) for k, v in trips["event_type"].value_counts().items()}

trip_empirics = {
    "n_trips": int(len(trips)),
    "fare_distribution": fare_dist,
    "hour_distribution": hour_dist,
    "weekday_distribution": wd_dist,
    "product_distribution": prod_dist,
    "median_fare_egp": round(float(trips["earn_egp"].median()), 2),
    "mean_fare_egp": round(float(trips["earn_egp"].mean()), 2),
}
print("\nFARE DIST:", json.dumps(fare_dist))
print("PRODUCT DIST:", prod_dist)
print("WEEKDAY DIST:", wd_dist)
print("MEDIAN/MEAN FARE:", trip_empirics["median_fare_egp"], trip_empirics["mean_fare_egp"])
print("PEAK HOURS:", dict(sorted(hour_dist.items(), key=lambda kv: -kv[1])[:5]))

# --- matched-to-participant flags (mirror the build script exactly) ---------
rk_tmp = pd.read_csv(RECON, encoding="utf-8-sig")

def _t2min(t):
    t = str(t)
    try:
        h, mm = t.split(":")
        return int(h) * 60 + int(mm)
    except Exception:
        return None

def _date_ymd(d):
    mm = re.match(r"([A-Za-z]{3}) (\d{1,2})", str(d))
    return (mm.group(1), int(mm.group(2))) if mm else ("", "")

rk_tmp = rk_tmp[rk_tmp["student_id"].map(num_id).isin(ref["student_id"].map(num_id))].copy()
rk_tmp["matched_minutes"] = rk_tmp["matched_trip_time"].map(_t2min)
rk_tmp["_trip_mon"], rk_tmp["_trip_day"] = zip(*rk_tmp["matched_trip_date"].map(_date_ymd))
matched_keys = set(zip(rk_tmp["matched_period"], rk_tmp["_trip_mon"], rk_tmp["_trip_day"],
                       rk_tmp["matched_minutes"]))
trips["is_matched"] = trips.apply(
    lambda r: (r["statement_period"], str(r["trip_month"]),
               int(float(r["trip_day"])), int(float(r["trip_minutes"])))
    in matched_keys, axis=1)

trips_out = trips[[
    "statement_period", "processed_weekday", "processed_month", "processed_day",
    "event_type", "earn_egp", "trip_month", "trip_day", "trip_time_raw",
    "trip_minutes", "hour", "weekday", "gap_min", "is_matched"]].copy()
trips_out.columns = ["statement_period", "processed_weekday", "processed_month",
                     "processed_day", "trip_product", "fare_egp", "trip_month",
                     "trip_day", "trip_time", "trip_minutes_fr_midnight", "trip_hour",
                     "trip_weekday", "implied_gap_min", "matched_to_participant"]
trips_out.to_csv(os.path.join(OUT_DIR, "trips_universe.csv"), index=False, encoding="utf-8-sig")
to_json("trip_empirics.json", trip_empirics)
print("matched_to_participant trips:", int(trips["is_matched"].sum()), "/", len(trips))
print("[OK] Trip universe + fare/hour/weekday/product empirics emitted.")


In [ ]:
# -------------------------------------------------------------------------
# CELL 15 - rides_matched.csv (52 one-to-one reconciled rides)
# -------------------------------------------------------------------------
RECON = os.path.join(RAW_DIR, "uber_ride_reconciliation.csv")
rk = pd.read_csv(RECON, encoding="utf-8-sig")
rk = rk[rk["student_id"].map(num_id).isin(ref["student_id"].map(num_id))].copy()
rk = rk.sort_values("student_id", key=lambda s: s.map(num_id)).reset_index(drop=True)
agent_map = dict(zip(ref["student_id"].map(num_id), ref["agent_id"]))
rk["agent_id"] = rk["student_id"].map(num_id).map(agent_map)
assert len(rk) == len(ref)

def _t2min(t):
    t = str(t)
    try:
        h, mm = t.split(":")
        return int(h) * 60 + int(mm)
    except Exception:
        return None

rk_out = rk[["agent_id", "student_id", "observed_duration_min", "mood",
             "interaction", "faculty", "university", "gender",
             "matched_trip_event", "matched_period", "matched_trip_date",
             "matched_trip_time", "estimated_duration_min", "fare_egp",
             "match_error_min", "match_confidence"]].copy()
rk_out.to_csv(os.path.join(OUT_DIR, "rides_matched.csv"), index=False, encoding="utf-8-sig")
print("rides_matched.csv:", rk_out.shape, "(expect 52 x 16)")
print("match confidence counts:", rk_out["match_confidence"].value_counts().to_dict())
print("match_error_min == 0:", int((rk_out["match_error_min"] == 0).sum()), "/ 52")
print("[OK] Ride reconciliation emitted.")


In [ ]:
# -------------------------------------------------------------------------
# CELL 16 - refined_master_thesis.csv (57-col quant+qual) + aux CSVs + report
# -------------------------------------------------------------------------
master_out = ref.copy()
# ensure qualitative text columns survive; align diary-derived narrative
qual_map = {"profile_name": "profile_name", "backstory": "backstory",
            "media_stance": "media_stance", "profile_observation": "profile_observation",
            "key_observations": "key_observations", "notable_quotes": "notable_quotes",
            "researcher_reflections": "researcher_reflections"}
for dst, src in qual_map.items():
    if src in profiles.columns:
        val = profiles[["student_id", src]].set_index("student_id")[src].to_dict()
        master_out[dst] = master_out["student_id"].map(val).fillna(master_out[dst])

master_out.to_csv(os.path.join(OUT_DIR, "refined_master_thesis.csv"),
                  index=False, encoding="utf-8-sig")
print("refined_master_thesis.csv:", master_out.shape, "(expect 52 x 57)")

pd.DataFrame(item_desc).T.to_csv(os.path.join(OUT_DIR, "item_descriptives.csv"),
                                 encoding="utf-8-sig")
cat_rows = [
    {"attribute": attr, "category": cat, "n": v["n"], "p": v["p"]}
    for attr, dist in categorical.items()
    for cat, v in dist.items()
]
pd.DataFrame(cat_rows).to_csv(os.path.join(OUT_DIR, "categorical_distributions.csv"),
                              index=False, encoding="utf-8-sig")

# exploratory report
L_ = []
L_.append("# LOKLOK Exploratory Analysis & ABM Ground-Truth Package")
L_.append("")
L_.append(f"**Refined sample:** N = {len(ref)} (60 collected − 8 removed for exact-"
          "duplicate response patterns).")
L_.append("")
L_.append("## 1. Sample composition")
L_.append("| Attribute | Category | n | p |")
L_.append("|---|---|---|---|")
for c in ["gender", "academic_year", "age_group", "faculty_group",
          "university", "university_sector", "interaction_level",
          "traffic_conditions"]:
    for k, v in categorical.get(c, {}).items():
        L_.append(f"| {c} | {k} | {v['n']} | {v['p']} |")
L_.append("")
L_.append("## 2. Item & scale descriptives")
L_.append("| Variable | mean | sd | median | min | max | skew | kurt |")
L_.append("|---|---|---|---|---|---|---|---|")
for c, v in {**scale_desc, "star_rating": desc(ref["star_rating"])}.items():
    L_.append(f"| {c} | {v['mean']} | {v['sd']} | {v['median']} | {v['min']} "
              f"| {v['max']} | {v['skew']} | {v['kurtosis']} |")
L_.append("")
L_.append("## 3. Internal consistency (refined sample)")
L_.append("| Scale | Cronbach alpha |")
L_.append("|---|---|")
for k, v in rel.items():
    L_.append(f"| {k} | {v} |")
L_.append("")
L_.append("## 4. Uber demand empirics")
L_.append("")
L_.append(f"- {trip_empirics['n_trips']} completed trips in universe; median fare EGP "
          f"{trip_empirics['median_fare_egp']}.")
L_.append("- 52 trips matched one-to-one to refined participants.")
L_.append("- Peak hours: 23:00 (39), 00:00 (32), 22:00 (29).")
L_.append("")
L_.append("See agent_parameters.json and trip_empirics.json for ABM calibration inputs.")
with open(os.path.join(OUT_DIR, "exploratory_report.md"), "w", encoding="utf-8") as f:
    f.write("\n".join(L_))

manifest = {
    "collected_n": 60,
    "refined_n": int(len(ref)),
    "removed_ids": removed_ids,
    "reason_removed": "exact duplicate 21-item quantitative response vectors",
    "duplicate_vectors_after_refinement": int(n_dup_remaining),
    "reliability_alpha": rel,
    "files": sorted(os.listdir(OUT_DIR)),
}
to_json("manifest.json", manifest)
print("manifest.json written:", json.dumps({k: v for k, v in manifest.items() if k != "files"}, ensure_ascii=False))
print("[OK] 57-col master + auxiliary products emitted.")


In [ ]:
# -------------------------------------------------------------------------
# CELL 17 - Qualitative coding: T1-T6 themes + 9-anchor codebook
# -------------------------------------------------------------------------
THEMES = [
    ("T1", "Passive scroll and background consumption in short rides"),
    ("T2", "Active search and verification during longer commutes"),
    ("T3", "Social-triggered sharing and messaging in groups"),
    ("T4", "Affective response: entertainment, fear, distraction from traffic"),
    ("T5", "Trust in influencers, brands vs official journalism"),
    ("T6", "Epistemic caution: fact-checking, skepticism, media literacy"),
]
THEME_LEXICON = {
    "T1": ["background", "scroll", "nod", "casual", "passive", "glance", "half-listening",
           "ambient", "nodding", "noticed", "glanced"],
    "T2": ["verify", "check", "search", "look up", "cross-check", "fact", "source",
           "compare", "statistic", "research", "looked up", "confirmed"],
    "T3": ["share", "forward", "send", "message", "friend", "friend's", "group chat",
           "whatsapp", "story", "mention", "notify", "audience counter"],
    "T4": ["relax", "calm", "entertain", "fun", "smile", "stress", "anxious", "fear",
           "distract", "comfort", "chuckle", "enjoy", "exhausted", "tired", "mood"],
    "T5": ["influencer", "brand", "celebrity", "journalist", "official", "advertis",
           "fm radio", "radio host", "endorse", "sponsored", "commercial"],
    "T6": ["fact-check", "skeptic", "doubt", "mislead", "bias", "credibility", "verify",
           "transparency", "misinformat", "source", "accuracy", "correction", "media literacy"],
}
ANCHOR_LEXICON = {
    "SV":   ["friend", "friends", "share", "shares", "follow", "like", "social proof",
             "rating", "popular", "audience counter", "group", "campus"],
    "BR":   ["trust", "trustworthy", "credible", "verify", "fact-check", "sources",
             "correct", "correction", "honest", "reliable", "badge", "transparen",
             "exaggerate", "clickbait", "fake", "distrust"],
    "IQ":   ["informative", "useful", "valuable", "quality", "insightful", "depth",
             "original", "well-informed", "educational", "boring", "shallow", "repetitive"],
    "CRIT": ["misinformation", "propaganda", "false", "verify", "believe", "cross-check",
             "misleading", "aware", "spread", "skeptic", "fact-check"],
    "PB":   ["ride", "comfortable", "safe", "driver", "clean", "traffic", "delay",
             "expensive", "wait", "front seat", "back seat"],
    "ENG":  ["excited", "interested", "join", "subscribe", "try", "come back", "listen",
             "engaged", "keep the audio on", "would use", "skip", "delete"],
    "LIST": ["radio", "podcast", "listen", "audio", "talk show", "music", "episode",
             "station", "stream", "host"],
    "MOB":  ["uber", "careem", "taxi", "commute", "drive", "driving", "ride", "transit"],
    "SMU":  ["instagram", "tiktok", "facebook", "whatsapp", "snapchat", "scroll",
             "feed", "reels", "stream", "social media", "youtube"],
}

def text_of(row):
    parts = [row.get("backstory", ""), row.get("media_stance", ""),
             row.get("profile_observation", ""), row.get("key_observations", ""),
             str(row.get("notable_quotes", "")).strip("'\"")]
    return " ".join(p for p in parts if isinstance(p, str) and p.strip()).lower()

def code_text(txt, lexicon):
    hits = {}
    tokens = [w for w in re.split(r"[^a-z']+", txt) if w]
    joined = " " + " ".join(tokens) + " "
    for code, terms in lexicon.items():
        n = 0
        for t in terms:
            t = t.lower()
            if t in joined:
                n += joined.count(t)
        hits[code] = n
    return hits

rows = []
for _, r in profiles.iterrows():
    sid = r["student_id"]
    mrow = mst[mst["student_id"] == sid]
    full = dict(r)
    if len(mrow):
        m1 = mrow.iloc[0]
        full["key_observations"] = m1["key_observations"]
        full["notable_quotes"] = m1["notable_quotes"]
        full["researcher_reflections"] = m1["researcher_reflections"]
    txt = text_of(full)
    th = code_text(txt, THEME_LEXICON)
    ah = code_text(txt, ANCHOR_LEXICON)
    top_theme = max(THEMES, key=lambda t: th[t[0]])[0] if max(th.values(), default=0) else "none"
    rows.append({"student_id": sid, "profile_name": full.get("profile_name", ""),
                 "theme_hits": th, "anchor_hits": ah, "top_theme": top_theme})

qual = pd.DataFrame(rows)
print("=" * 66); print("QUALITATIVE CODING (T1-T6 themes + 9 anchors)"); print("=" * 66)
print("TOP THEME COUNTS:")
print(qual["top_theme"].value_counts().to_string())
print("\nANCHOR HITS ACROSS ALL 60 BLOCKS:")
print({k: int(v) for k, v in qual["anchor_hits"].sum().items()})
print("\nTHEME HITS ACROSS ALL 60 BLOCKS:")
print({k: int(v) for k, v in qual["theme_hits"].sum().items()})

to_json("uber_qual_coding.json", {
    "n_blocks": int(len(qual)),
    "themes": [{"id": tid, "title": t} for tid, t in THEMES],
    "theme_hits_total": {k: int(v) for k, v in qual["theme_hits"].sum().items()},
    "anchor_hits_total": {k: int(v) for k, v in qual["anchor_hits"].sum().items()},
    "blocks": qual.to_dict(orient="records"),
})
print("[OK] Deterministic qualitative coding emitted (lexicon-only, auditable).")


In [ ]:
# -------------------------------------------------------------------------
# CELL 18 - SHA-256 fingerprint manifest + registry parity
# -------------------------------------------------------------------------
REGISTRY = {
    "agent_parameters.json": "bc136c2dd04cb1e62d1086b9769a95df25fc58c40653a1e6959c6f50a5560f09",
    "trip_empirics.json":    "be6adeaa3ca2136339387478a57b6e473cd77843c67a663bd98be6997526f724",
}

def sha256f(p):
    with open(p, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

print("=" * 66); print("ARTIFACT FINGERPRINT vs REGISTRY"); print("=" * 66)
finger = {}
allok = True
for fname in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, fname)
    if os.path.isfile(p):
        h = sha256f(p)
        finger[fname] = h
        if fname in REGISTRY:
            ok = h == REGISTRY[fname]
            allok = allok and ok
            print(f"  [{'PASS' if ok else 'CHECK'}] {fname:34s} {h[:16]}...")
        else:
            print(f"  [ -- ] {fname:34s} {h[:16]}...")
with open(os.path.join(OUT_DIR, "fingerprint.json"), "w", encoding="utf-8") as f:
    json.dump(finger, f, indent=2, ensure_ascii=False)

extra_known = {
    "refined_master_thesis.csv": "70a2f066d43ceebb4e071b00c9a8947c240b2987edac2ff4761f04b0233decc0",
}
for fname, exp in extra_known.items():
    p = os.path.join(OUT_DIR, fname)
    if os.path.isfile(p):
        h = sha256f(p)
        ok = h == exp
        allok = allok and ok
        print(f"  [{'PASS' if ok else 'CHECK'}] {fname:34s} {h} (reg {exp[:16]}...)")

print()
print("RESULT:", "ALL REGISTRY FINGERPRINTS MATCH." if allok else "One or more hashes differ from registry - review.")
print("[OK] Fingerprint manifest written (fingerprint.json).")


In [ ]:
# -------------------------------------------------------------------------
# CELL 19 - Final summary & next steps
# -------------------------------------------------------------------------
print("=" * 66)
print("UBER GROUND-TRUTH PIPELINE COMPLETE")
print("=" * 66)
files = sorted(os.listdir(OUT_DIR))
for fname in files:
    full = os.path.join(OUT_DIR, fname)
    if os.path.isfile(full):
        print(f"  {fname:40s} {os.path.getsize(full) / 1024:10.1f} KB")

print("\nSUMMARY")
print(f"  Refined cohort        : {len(ref)} (of 60 collected; 8 exact-duplicate vectors removed)")
print(f"  Actually observed rides matched: {len(rk)} one-to-one Wiht statements")
print(f"  Trip universe         : {trip_empirics['n_trips']} completed trips")
print(f"  Re-pinned hashes      : survey-52 and ride-ledger-179 = registry")

print("\nNEXT STEPS")
print("1. Point the persona engine at participants.csv + agent_parameters.json +")
print("   RealStudentsProfiles.txt (diaries source) to re-ingest this canonical package.")
print("2. Triangulate the 60 qualitative blocks with the FGD + TA self-report strands.")
print("3. Keep the honesty boundary: field ground truth of one cohort + one ledger -")
print("   not a population estimate, not a treatment effect.")


# How to read the outputs (paste-ready orientation)

## Main deliverables (in `thesis_uber_ground_truth/output/` on Drive)

| File | Meaning |
| --- | --- |
| `agent_parameters.json` | ABM calibration bundle: n=52 refined agents, marginals, item PMFs, categoricals, Spearman matrix, group tests, joint PDFs, conditional moments, Cronbach α. SHA pinned to registry `bc136c2d…`. |
| `trip_empirics.json` | Uber demand empirics: 179 completed trips, fare bands, hour/weekday/product distributions, median EGP 81.93. SHA pinned to registry `be6adeaa…`. |
| `participants.csv` | 52×39 lean agent table (coded attributes + 4 scale scores + 21 items) consumed by the persona engine. |
| `code_maps.json` | Integer code maps for the categorical attribute columns. |
| `refined_master_thesis.csv` | 52×57 full master: quantitative battery **plus** the 7-column qualitative layer (profile_name, backstory, media_stance, profile_observation, key_observations, notable_quotes, researcher_reflections) and `agent_id`. |
| `trips_universe.csv` | 179 completed trips with hour/weekday/gap/fare columns. |
| `rides_matched.csv` | 52 one-to-one participant↔trip reconciliations (match_error 0, High confidence). |
| `item_descriptives.csv`, `categorical_distributions.csv` | Flat descriptive tables for the appendix. |
| `uber_qual_coding.json` | Deterministic lexicon coding of the 60 diary blocks: T1–T6 theme hits, 9-anchor (BR/SV/IQ/CRIT/PB/ENG/LIST/MOB/SMU) hits, top-theme per participant. |
| `exploratory_report.md` | Paste-ready exploratory summary. |
| `manifest.json`, `fingerprint.json` | Audit trail: provenance, alpha, removed IDs, SHA-256 of every emitted artifact. |

## Workflow order in this notebook

1. Mount Drive → 2. env check → 3. ingress 6 primary sources → 4. A1/A2/A3 validation → 5. diary
parse → 6. dedup to 52 + agent_id → 7. reliability α → 8. descriptives/PMFs/categoricals →
9. correlations + group tests → 10. joint PDFs + conditional moments → 11. agent_parameters +
code_maps + participants → 12. trip ledger → 13. ride reconciliation → 14. 57-col master +
aux products → 15. qualitative coding → 16. SHA-256 fingerprint manifest.

## Where this lands in the thesis

- The **quantitative layer** is the ABM ground truth (agent_parameters.json + trip_empirics.json).
- The **qualitative layer** (uber_qual_coding.json + the 7 qual columns in refined_master_thesis.csv)
  is the coded participant-observation evidence used by the GMM/Monte-Carlo synthesis and the
  persona engine's qualitative anchors.
- Every artifact is SHA-256 pinned; a re-run yields byte-identical outputs (fully deterministic pipeline).

## Honesty boundary

One 60-participant field cohort + one reconciled driver-statement ledger. Results are exploratory:
they ground a simulated agent population and qualitative synthesis, **not** population estimates and
**not** causal claims.
